# Generate SPICE_F0P_V6.7B.R1 Signal

Generates the MISO `TEST.TEST.Signal.MISO.SPICE_F0P_V6.7B.R1` signal for **all tradable period types and class types** using ML predictions from the shadow price prediction pipeline.

**Signal spec:**
- Base signal: `TEST.TEST.Signal.MISO.SPICE_F0P_V6.2B.R1`
- Period types: All tradable per auction month (f0, f1, f2, f3, q2, q3, q4)
- Class types: onpeak, offpeak
- Ranking: 0.4 × prob_rank + 0.3 × hist_shadow_rank + 0.3 × pred_shadow_rank
- Tiers: 5 quantile bins (0 = best, 4 = worst)

In [ ]:
from pbase.config.ray import init_ray
import pmodel
import shadow_price_prediction

init_ray(address='ray://10.8.0.36:10001', extra_modules=[pmodel, shadow_price_prediction])

In [ ]:
import gc
import logging
import resource
from pathlib import Path

import pandas as pd
from pbase.analysis.tools.all_positions import MisoApTools

from shadow_price_prediction import (
    ShadowPricePipeline,
    PredictionConfig,
    FeatureConfig,
    ModelConfig,
    ModelSpec,
    EnsembleConfig,
    TrainingConfig,
    ThresholdConfig,
    AnomalyDetectionConfig,
    XGBClassifier, XGBRegressor,
    LogisticRegression, ElasticNet,
    aggregate_multi_month_results,
    generate_and_save_signal,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def mem_mb():
    """Current process peak RSS in MB."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


print(f"Imports done. Memory: {mem_mb():.0f} MB")

## 1. Configuration

In [ ]:
# === Signal Configuration ===
RTO = 'miso'
SIGNAL_NAME = 'TEST.TEST.Signal.MISO.SPICE_F0P_V6.7B.R1'
MARKET_ROUND = 1

# Auction months to generate signals for
AUCTION_MONTHS = pd.date_range('2024-06', '2026-02', freq='MS')

# Class types to generate
CLASS_TYPES = ['onpeak', 'offpeak']

# Set to True for a test run without actually saving
DRY_RUN = False

# Output directory for intermediate ML results (speeds up reruns)
OUTPUT_DIR = '/opt/temp/tmp/pw_data/spice6/signal_67b_results'

# Initialize MISO AP Tools for period type lookups
aptools = MisoApTools()

In [ ]:
# === Feature Configuration ===
# Format: (feature_name, monotonicity_constraint)
# 1 = increasing, -1 = decreasing, 0 = no constraint
#
# Available columns in training data (from data_loader):
#   prob_exceed_{80,85,90,95,100,105,110}  — P(flow > X% of limit)
#   prob_below_{80,85,90,95,100,105,110}   — P(flow < X% of limit)
#   density_{mean,variance,skewness,kurtosis}
#   hist_da, recent_hist_da, season_hist_da_{1,2,3}

# Step 1 (classification): probability and density shape features
step1_features = [
    ('prob_exceed_110', 1),
    ('prob_exceed_105', 1),
    ('prob_exceed_100', 1),
    ('prob_exceed_95', 1),
    ('prob_exceed_90', 1),
    ('prob_below_95', -1),
    ('prob_below_90', -1),
    ('density_skewness', 1),
]

# Step 2 (regression): same probability features + historical shadow price info
step2_features = [
    ('prob_exceed_110', 1),
    ('prob_exceed_105', 1),
    ('prob_exceed_100', 1),
    ('prob_exceed_95', 1),
    ('prob_exceed_90', 1),
    ('prob_below_95', -1),
    ('prob_below_90', -1),
    ('density_skewness', 1),
    ('hist_da', 1),
    ('recent_hist_da', 1),
]

feature_config = FeatureConfig(
    step1_features=step1_features,
    step2_features=step2_features,
)
print(f"Step1: {len(step1_features)} features, Step2: {len(step2_features)} features")
print(f"All features (union): {feature_config.all_features}")

In [ ]:
# === Model Configuration (Standard Ensemble) ===
xgb_clf_spec = ModelSpec(
    model_class=XGBClassifier,
    config=ModelConfig(params={
        'n_estimators': 200,
        'max_depth': 4,
        'learning_rate': 0.1,
        'n_jobs': 1,
        'eval_metric': 'logloss'
    }),
    weight=0.5
)

lr_clf_spec = ModelSpec(
    model_class=LogisticRegression,
    config=ModelConfig(params={
        'C': 1.0,
        'max_iter': 1000,
        'class_weight': 'balanced',
        'n_jobs': 1
    }),
    weight=0.5
)

xgb_reg_spec = ModelSpec(
    model_class=XGBRegressor,
    config=ModelConfig(params={
        'n_estimators': 200,
        'max_depth': 4,
        'learning_rate': 0.1,
        'n_jobs': 1
    }),
    weight=0.5
)

enet_reg_spec = ModelSpec(
    model_class=ElasticNet,
    config=ModelConfig(params={
        'alpha': 1.0,
        'l1_ratio': 0.5
    }),
    weight=0.5
)

ensemble_config = EnsembleConfig(
    default_classifiers=[xgb_clf_spec, lr_clf_spec],
    branch_classifiers=[xgb_clf_spec, lr_clf_spec],
    default_regressors=[xgb_reg_spec, enet_reg_spec],
    branch_regressors=[xgb_reg_spec, enet_reg_spec]
)

## 2. Helper Functions

In [ ]:
def get_market_months_for_period(auction_month: pd.Timestamp, period_type: str) -> list[pd.Timestamp]:
    """Get market months for a given auction month and period type."""
    market_st, market_et = aptools.tools.get_market_month_from_auction_month_and_period_trades(
        auction_month=auction_month,
        period_type=period_type,
    )
    return list(pd.date_range(market_st, market_et, freq='MS', inclusive='left'))


def run_pipeline_for_period(
    auction_month: pd.Timestamp,
    period_type: str,
    class_type: str,
) -> pd.DataFrame | None:
    """Run the ML pipeline for a single (auction_month, period_type, class_type).
    
    Returns the aggregated final_results DataFrame, or None on failure.
    """
    market_months = get_market_months_for_period(auction_month, period_type)
    
    if not market_months:
        logger.warning(f"No market months for {auction_month} {period_type}")
        return None
    
    # Check if cached results exist for all market months
    cached_results = []
    all_cached = True
    auc_str = auction_month.strftime('%Y-%m')
    
    for mm in market_months:
        mm_str = mm.strftime('%Y-%m')
        cached_path = Path(OUTPUT_DIR) / f'auction_month={auc_str}/market_month={mm_str}/class_type={class_type}/final_results.parquet'
        if cached_path.exists():
            cached_results.append(pd.read_parquet(cached_path))
        else:
            all_cached = False
            break
    
    if all_cached:
        logger.info(f"Using cached results for {auc_str} {period_type} {class_type} ({len(cached_results)} months)")
        if len(cached_results) == 1:
            return cached_results[0]
        return aggregate_multi_month_results(cached_results)
    
    # Build test periods
    test_periods = [(auction_month, mm) for mm in market_months]
    
    # Create pipeline config — paths come from default MISO_ISO_CONFIG
    # TrainingConfig: 6 months fit, 3 months validation, 3 months holdout (12 total lookback)
    config = PredictionConfig(
        market_round=MARKET_ROUND,
        period_type=period_type,
        class_type=class_type,
        features=feature_config,
        models=ensemble_config,
        training=TrainingConfig(
            min_samples_for_branch_model=10,
            train_months=6,
            val_months=3,
            test_months=3,
        ),
        threshold=ThresholdConfig(threshold_beta=2.0),
        anomaly_detection=AnomalyDetectionConfig(enabled=True),
    )
    
    pipeline = ShadowPricePipeline(config)
    
    try:
        results_per_outage, final_results_combined, metrics, train_data_dict, predictors = pipeline.run(
            test_periods=test_periods,
            class_type=class_type,
            verbose=True,
            use_parallel=True,
            n_jobs=2,
            output_dir=OUTPUT_DIR,
        )
    except Exception as e:
        logger.error(f"Pipeline failed for {auc_str} {period_type} {class_type}: {e}")
        return None
    finally:
        # Free pipeline memory immediately
        del pipeline
        gc.collect()
        print(f"  [mem] after pipeline run: {mem_mb():.0f} MB")
    
    # For multi-month periods, collect per-month final_results and aggregate
    if len(market_months) > 1:
        month_results = []
        for mm in market_months:
            mm_str = mm.strftime('%Y-%m')
            cached_path = Path(OUTPUT_DIR) / f'auction_month={auc_str}/market_month={mm_str}/class_type={class_type}/final_results.parquet'
            if cached_path.exists():
                month_results.append(pd.read_parquet(cached_path))
        if month_results:
            return aggregate_multi_month_results(month_results)
    
    # Single month — return combined directly
    if final_results_combined is not None and not final_results_combined.empty:
        return final_results_combined
    
    return None

## 3. Generate Signals

Loop over all auction months, period types, and class types.

In [ ]:
results_log = []
print(f"Starting signal generation. Memory: {mem_mb():.0f} MB")

for auction_month in AUCTION_MONTHS:
    auc_str = auction_month.strftime('%Y-%m')
    
    # Get tradable period types for this auction month
    tradable_periods = aptools.tools.get_tradable_period_types(auction_month)
    print(f"\n{'='*80}")
    print(f"Auction Month: {auc_str} | Memory: {mem_mb():.0f} MB")
    print(f"Tradable periods: {tradable_periods}")
    print(f"{'='*80}")
    
    for class_type in CLASS_TYPES:
        for period_type in tradable_periods:
            print(f"\n--- {auc_str} | {period_type} | {class_type} ---")
            
            final_results = None
            signal_df = None
            try:
                final_results = run_pipeline_for_period(
                    auction_month=auction_month,
                    period_type=period_type,
                    class_type=class_type,
                )
                
                if final_results is None or final_results.empty:
                    logger.warning(f"No results for {auc_str} {period_type} {class_type}")
                    results_log.append({
                        'auction_month': auc_str,
                        'period_type': period_type,
                        'class_type': class_type,
                        'status': 'no_results',
                        'n_constraints': 0,
                    })
                    continue
                
                signal_df, save_path = generate_and_save_signal(
                    final_results=final_results,
                    rto=RTO,
                    signal_name=SIGNAL_NAME,
                    period_type=period_type,
                    class_type=class_type,
                    auction_month=auction_month,
                    dry_run=DRY_RUN,
                )
                
                status = 'dry_run' if DRY_RUN else 'saved'
                tier_dist = signal_df['tier'].value_counts().sort_index().to_dict() if not signal_df.empty else {}
                
                results_log.append({
                    'auction_month': auc_str,
                    'period_type': period_type,
                    'class_type': class_type,
                    'status': status,
                    'n_constraints': len(signal_df),
                    'tier_distribution': tier_dist,
                    'path': save_path,
                })
                
                print(f"  => {len(signal_df)} constraints, tiers: {tier_dist}")
                
            except Exception as e:
                logger.error(f"Failed: {auc_str} {period_type} {class_type}: {e}")
                results_log.append({
                    'auction_month': auc_str,
                    'period_type': period_type,
                    'class_type': class_type,
                    'status': f'error: {e}',
                    'n_constraints': 0,
                })
            finally:
                del final_results, signal_df
                gc.collect()

print(f"\nSignal generation complete. Memory: {mem_mb():.0f} MB")

## 4. Summary

In [ ]:
log_df = pd.DataFrame(results_log)
print(f"\nTotal signals generated: {len(log_df[log_df['status'].isin(['saved', 'dry_run'])])}")
print(f"Failures: {len(log_df[log_df['status'].str.startswith('error')])}")
print(f"No results: {len(log_df[log_df['status'] == 'no_results'])}")
print()

# Coverage matrix
if not log_df.empty:
    coverage = log_df.pivot_table(
        index='auction_month',
        columns=['period_type', 'class_type'],
        values='n_constraints',
        aggfunc='first',
        fill_value=0,
    )
    print("Coverage matrix (constraint count per signal):")
    display(coverage)

## 5. Validation

### Layer 1: Format & Schema Validation
Verify signal format, index structure, and column values for a sample signal.

In [ ]:
from pbase.data.dataset.signal.general import ConstraintsSignal

EXPECTED_COLUMNS = {
    'constraint_id', 'branch_name', 'flow_direction', 'hist_da',
    'predicted_shadow_price', 'binding_probability_scaled',
    'prob_rank', 'hist_shadow_rank', 'pred_shadow_rank', 'rank', 'tier',
    'shadow_sign', 'shadow_price', 'equipment',
}

test_am = pd.Timestamp('2025-01')
validation_errors = []

# Validate all saved signals for the test auction month
saved_signals = [r for r in results_log if r['auction_month'] == '2025-01' and r['status'] in ('saved', 'dry_run') and r['n_constraints'] > 0]

for entry in saved_signals:
    pt, ct = entry['period_type'], entry['class_type']
    label = f"2025-01/{pt}/{ct}"
    try:
        sig = ConstraintsSignal(rto='miso', signal_name=SIGNAL_NAME, period_type=pt, class_type=ct).load_data(test_am)
        
        errors = []
        if set(sig.columns) != EXPECTED_COLUMNS:
            errors.append(f"Column mismatch: got {set(sig.columns) - EXPECTED_COLUMNS} extra, missing {EXPECTED_COLUMNS - set(sig.columns)}")
        if not sig.index.str.contains('|', regex=False).all():
            errors.append("Index missing '|' separator")
        if not sig.index.str.endswith('|spice').all():
            errors.append("Index not ending with '|spice'")
        if not sig['tier'].isin([0, 1, 2, 3, 4]).all():
            errors.append(f"Invalid tier values: {sig['tier'].unique()}")
        if not (sig['predicted_shadow_price'] > 0).all():
            errors.append("predicted_shadow_price has non-positive values")
        if not sig['shadow_sign'].isin([-1, 1]).all():
            errors.append(f"Invalid shadow_sign values: {sig['shadow_sign'].unique()}")
        if not (sig['equipment'] == sig['branch_name']).all():
            errors.append("equipment != branch_name")
        
        if errors:
            validation_errors.extend([(label, e) for e in errors])
            print(f"  FAIL {label}: {errors}")
        else:
            print(f"  OK   {label}: {len(sig)} constraints, tiers={sig['tier'].value_counts().sort_index().to_dict()}")
    except Exception as e:
        validation_errors.append((label, str(e)))
        print(f"  ERR  {label}: {e}")

if validation_errors:
    print(f"\n{len(validation_errors)} validation errors found!")
    for label, err in validation_errors:
        print(f"  {label}: {err}")
else:
    print(f"\nAll {len(saved_signals)} signals passed schema validation.")

### Layer 2: Cross-Reference with 6.2B

Compare 6.7B signals against the existing 6.2B signal for overlapping period types to verify consistency.

In [ ]:
SIGNAL_62B = 'TEST.TEST.Signal.MISO.SPICE_F0P_V6.2B.R1'
CROSS_REF_PAIRS = [('f0', 'onpeak'), ('f0', 'offpeak'), ('f1', 'onpeak'), ('f1', 'offpeak'), ('q4', 'onpeak'), ('q4', 'offpeak')]
cross_ref_am = pd.Timestamp('2025-01')

cross_ref_results = []
for pt, ct in CROSS_REF_PAIRS:
    label = f"{pt}/{ct}"
    try:
        sig_62b = ConstraintsSignal('miso', SIGNAL_62B, pt, ct).load_data(cross_ref_am)
        sig_67b = ConstraintsSignal('miso', SIGNAL_NAME, pt, ct).load_data(cross_ref_am)
    except Exception as e:
        print(f"  SKIP {label}: {e}")
        continue

    ids_62b = set(sig_62b['constraint_id'])
    ids_67b = set(sig_67b['constraint_id'])
    overlap = ids_67b & ids_62b
    overlap_pct = len(overlap) / max(len(ids_67b), 1) * 100

    # Tier correlation for overlapping constraints
    tier_corr = float('nan')
    if overlap:
        m = sig_67b[['constraint_id', 'tier']].merge(
            sig_62b[['constraint_id', 'tier']], on='constraint_id', suffixes=('_67b', '_62b')
        )
        if len(m) > 1:
            tier_corr = m['tier_67b'].corr(m['tier_62b'])

    # Shadow price distribution comparison
    sp_mean_67b = sig_67b['predicted_shadow_price'].mean()
    sp_mean_62b = sig_62b['predicted_shadow_price'].mean()

    row = {
        'period_type': pt, 'class_type': ct,
        'n_67b': len(sig_67b), 'n_62b': len(sig_62b),
        'overlap': len(overlap), 'overlap_pct': f"{overlap_pct:.0f}%",
        'tier_corr': f"{tier_corr:.3f}",
        'sp_mean_67b': f"{sp_mean_67b:.1f}", 'sp_mean_62b': f"{sp_mean_62b:.1f}",
    }
    cross_ref_results.append(row)
    print(f"  {label}: 6.7B={len(sig_67b)}, 6.2B={len(sig_62b)}, overlap={len(overlap)} ({overlap_pct:.0f}%), tier_corr={tier_corr:.3f}")

if cross_ref_results:
    print("\nCross-reference summary:")
    display(pd.DataFrame(cross_ref_results))

### Layer 3: evaluate_signal_v2 Backtesting

Run `evaluate_signal_v2()` per (period_type, class_type) to measure precision, recall, and F1 against actual DA shadow prices.

**Acceptance criteria:**
- `recall,t~0` > 0.30 for monthly periods, > 0.20 for quarterly
- `prec,t~0` > 0.15 for all periods
- `sp%,t~0` > 0.40 (tier 0 captures most impactful constraints)
- No period type should have 0 constraints or all-NaN metrics

In [ ]:
EVAL_PERIODS = ['f0', 'f1', 'f2', 'f3', 'q2', 'q3', 'q4']
EVAL_CLASSES = ['onpeak', 'offpeak']
EVAL_AM_ST = '2024-06'
EVAL_AM_ET = '2025-12'

eval_results = {}
for period_type in EVAL_PERIODS:
    for class_type in EVAL_CLASSES:
        label = f"{period_type}/{class_type}"
        try:
            metrics_df, for_recall, for_precision, signal_dict, _ = aptools.tools.evaluate_signal_v2(
                signal_name=SIGNAL_NAME,
                period_type=period_type,
                peak_type=class_type,
                auction_month_st=EVAL_AM_ST,
                auction_month_et_in=EVAL_AM_ET,
                n_jobs=12,
            )
            eval_results[label] = metrics_df
            print(f"\n{label}:")
            display(metrics_df[['auction_month', 'sp', 'sp%,t~0', 'recall,t~0', 'prec,t~0', 'F1,t~0']])
        except Exception as e:
            print(f"\n{label}: FAILED — {e}")
            eval_results[label] = None

print(f"\nMemory after evaluation: {mem_mb():.0f} MB")

In [ ]:
# Acceptance criteria check
MONTHLY_PERIODS = {'f0', 'f1', 'f2', 'f3'}
QUARTERLY_PERIODS = {'q2', 'q3', 'q4'}
acceptance_failures = []

for label, mdf in eval_results.items():
    if mdf is None:
        acceptance_failures.append((label, "evaluation failed"))
        continue
    pt = label.split('/')[0]
    is_quarterly = pt in QUARTERLY_PERIODS

    # Average across auction months (skip NaN)
    avg_recall = mdf['recall,t~0'].mean()
    avg_prec = mdf['prec,t~0'].mean()
    avg_sp_pct = mdf['sp%,t~0'].mean()

    recall_thresh = 0.20 if is_quarterly else 0.30
    if avg_recall < recall_thresh:
        acceptance_failures.append((label, f"recall,t~0 = {avg_recall:.3f} < {recall_thresh}"))
    if avg_prec < 0.15:
        acceptance_failures.append((label, f"prec,t~0 = {avg_prec:.3f} < 0.15"))
    if avg_sp_pct < 0.40:
        acceptance_failures.append((label, f"sp%,t~0 = {avg_sp_pct:.3f} < 0.40"))

if acceptance_failures:
    print(f"WARNING: {len(acceptance_failures)} acceptance criteria not met:")
    for label, msg in acceptance_failures:
        print(f"  {label}: {msg}")
else:
    print("All acceptance criteria passed.")

## 6. Cleanup

In [ ]:
import ray
ray.shutdown()
gc.collect()
print(f"Ray shutdown. Final memory: {mem_mb():.0f} MB")